# Retail Orders — Cleaning & Loading

Clean the raw orders file, add price and profit columns, and load it into SQL Server for analysis.

In [1]:
# Download (run once, needs a Kaggle API token)
# !kaggle datasets download ankitbansal06/retail-orders -f orders.csv

In [2]:
import zipfile

with zipfile.ZipFile('orders.csv.zip') as z:
    z.extractall()

## Load and inspect

In [3]:
import pandas as pd

df = pd.read_csv('orders.csv', na_values=['Not Available', 'unknown'])
print(df.shape)
df.isna().sum()

(9994, 16)


Order Id            0
Order Date          0
Ship Mode           6
Segment             0
Country             0
City                0
State               0
Postal Code         0
Region              0
Category            0
Sub Category        0
Product Id          0
cost price          0
List Price          0
Quantity            0
Discount Percent    0
dtype: int64

In [4]:
df.head()

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


## Clean column names and types

In [5]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d')
df.dtypes

order_id                     int64
order_date          datetime64[ns]
ship_mode                   object
segment                     object
country                     object
city                        object
state                       object
postal_code                  int64
region                      object
category                    object
sub_category                object
product_id                  object
cost_price                   int64
list_price                   int64
quantity                     int64
discount_percent             int64
dtype: object

## Remove rows with no price
507 rows (~5%) have both cost and list price at 0. They carry no money information, so I dropped them.

In [6]:
no_price = (df['list_price'] == 0) & (df['cost_price'] == 0)
print('Rows with no price:', no_price.sum())

df = df[~no_price].copy()
print('Rows left:', len(df))

Rows with no price: 507
Rows left: 9487


## Add price and profit columns
Per unit: discount, sale price, profit. Per order line: revenue and total profit (× quantity).

In [7]:
df['discount'] = df['list_price'] * df['discount_percent'] * 0.01
df['sale_price'] = df['list_price'] - df['discount']
df['profit'] = df['sale_price'] - df['cost_price']

df['revenue'] = df['sale_price'] * df['quantity']
df['total_profit'] = df['profit'] * df['quantity']

df[['list_price', 'discount_percent', 'sale_price', 'cost_price', 'profit', 'quantity', 'revenue']].head()

,list_price,discount_percent,sale_price,cost_price,profit,quantity,revenue
0,260,2,254.8,240,14.8,2,509.6
1,730,3,708.1,600,108.1,3,2124.3
2,10,5,9.5,10,-0.5,2,19.0
3,960,2,940.8,780,160.8,5,4704.0
4,20,5,19.0,20,-1.0,2,38.0


## Sanity checks

In [8]:
print('Duplicate order ids:', df['order_id'].duplicated().sum())
print('Date range:', df['order_date'].min().date(), 'to', df['order_date'].max().date())
print('Lines sold below cost:', (df['profit'] < 0).sum())
print('Total revenue:', round(df['revenue'].sum()))
print('Total profit:', round(df['total_profit'].sum()))
df['ship_mode'].value_counts(dropna=False)

Duplicate order ids: 0
Date range: 2022-01-01 to 2023-12-31
Lines sold below cost: 3997
Total revenue: 11079328
Total profit: 1039928


ship_mode
Standard Class    5637
Second Class      1869
First Class       1450
Same Day           525
NaN                  6
Name: count, dtype: int64

42% of lines sell below cost — but list price equals cost on 3,955 lines, so any discount makes them negative. Checked further in SQL.

## Load into SQL Server
Run `CREATE DATABASE retail_db;` in SSMS first.

In [9]:
import sqlalchemy as sal

engine = sal.create_engine(
    'mssql+pyodbc://@DESKTOP-G66G5TF/retail_db'
    '?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
)

df.to_sql('df_orders', con=engine, index=False, if_exists='replace')
print('Loaded', len(df), 'rows')

Loaded 9487 rows
